Linear Regression



In [1]:
!pip install m2cgen

In [2]:
!pip install mpy-cross

  Using cached mpy_cross-1.28.0.post2-py2.py3-none-manylinux1_x86_64.whl.metadata (4.0 kB)
Using cached mpy_cross-1.28.0.post2-py2.py3-none-manylinux1_x86_64.whl (1.1 MB)


In [3]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error, mean_absolute_percentage_error
from pathlib import Path
import os
import hashlib
import m2cgen as m2c
import subprocess
from sklearn.decomposition import PCA

In [4]:
from google.colab import drive
# 1. Collega Drive
drive.mount('/content/drive')

# 2. Vai nella cartella dove hai i file e la cartella audio
%cd "/content/drive/MyDrive/Magistrale/Tesi/Fase2"

Mounted at /content/drive
/content/drive/MyDrive/Magistrale/Tesi/Fase2


In [5]:
SEEDS = [42, 8, 1291, 64207, 305, 91876, 12, 456, 33920, 7]

In [6]:
CSV_INPUT_PATH = f"audio_dataset.csv"
LABEL_COLUMN="portata"
MODELS_DIR = Path(f"ModelsToEvaluate")
BEST_MODELS_CSV = Path(f"best_models_summary.csv")

MODELS_DIR.mkdir(parents=True, exist_ok=True)
#IMPORTANCES_DIR=Path("Importances")
#IMPORTANCES_DIR.mkdir(parents=True, exist_ok=True)
MODEL_NAME=f"LinearRegression"
START_FROM_SEED_INDEX=0

In [7]:
# 1. CARICAMENTO DATI
# Assumiamo che il file CSV abbia la colonna identificativa come prima colonna
df = pd.read_csv(CSV_INPUT_PATH, index_col=0)

In [8]:
# 2. PREPARAZIONE X e y
target_column = 'portata'
X = df.drop(columns=[target_column])
y = df[target_column]

In [9]:
'''
importances = pd.Series(lr_model.coef_, index=X_train.columns).abs()
'''

'\nimportances = pd.Series(lr_model.coef_, index=X_train.columns).abs()\n'

In [10]:
'''
plt.figure(figsize=(10, 6))
importances.nlargest(10).sort_values(ascending=True).plot(kind='barh', color='skyblue')

plt.title("Top 10 Feature per la stima (Linear Regression)")
plt.xlabel("Peso del Coefficiente (Impatto Assoluto)")
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
#plt.savefig(IMPORTANCES_DIR / f"{MODEL_NAME}_importances.png", dpi=300, bbox_inches='tight')
plt.show()
'''

'\nplt.figure(figsize=(10, 6))\nimportances.nlargest(10).sort_values(ascending=True).plot(kind=\'barh\', color=\'skyblue\')\n\nplt.title("Top 10 Feature per la stima (Linear Regression)")\nplt.xlabel("Peso del Coefficiente (Impatto Assoluto)")\nplt.grid(axis=\'x\', linestyle=\'--\', alpha=0.7)\nplt.tight_layout()\n#plt.savefig(IMPORTANCES_DIR / f"{MODEL_NAME}_importances.png", dpi=300, bbox_inches=\'tight\')\nplt.show()\n'

In [11]:
for i, seed in enumerate(SEEDS[START_FROM_SEED_INDEX:], start=START_FROM_SEED_INDEX):

  if START_FROM_SEED_INDEX != 0:
      print(f"Skipping the first {START_FROM_SEED_INDEX} seeds...")

  print(f"\n\n=== INIZIO TRAINING LINEAR REGRESSION CON SEME {seed} ({i+1}/{len(SEEDS)}) ===\n")

  # 3. SPLIT TRAIN/TEST
  # Dividiamo i dati: 80% training, 20% test
  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=seed)

  train_indices = df.index.get_indexer(X_train.index).tolist()
  test_indices = df.index.get_indexer(X_test.index).tolist()
  print("SHA256 Train set: "+hashlib.sha256(str(train_indices).encode('utf-8')).hexdigest())
  print("SHA256 Test set: "+hashlib.sha256(str(test_indices).encode('utf-8')).hexdigest())

  scaler = StandardScaler()
  scaler.set_output(transform="pandas")
  X_train = scaler.fit_transform(X_train)
  X_test = scaler.transform(X_test)

  pca = PCA(n_components=0.90, random_state=seed)
  pca.set_output(transform="pandas")

  print(f"Feature originali: {X_train.shape[1]}")

  # Trasforma i dati (X_train e X_test erano già stati scalati col tuo scaler)
  X_train = pca.fit_transform(X_train)
  X_test = pca.transform(X_test)

  print(f"Feature dopo PCA: {X_train.shape[1]}")

  # 2. Inizializzazione e Fit
  lr_model = LinearRegression()
  lr_model.fit(X_train, y_train)
  print("Training completato")

  y_train_pred = lr_model.predict(X_train)

  mae_training = mean_absolute_error(y_train, y_train_pred)
  mape_training = mean_absolute_percentage_error(y_train, y_train_pred) * 100
  r2_training = r2_score(y_train, y_train_pred)
  mse_training=mean_squared_error(y_train, y_train_pred)
  rmse_training = np.sqrt(mse_training)

  print(f"--- PERFORMANCE LINEAR REGRESSION (TRAINING) ---")
  print(f"R^2 Score: {r2_training:.4f}")
  print(f"MAE: {mae_training:.4f}")
  print(f"MAPE: {mape_training:.2f}%")
  print(f"MSE: {mse_training:.4f}")
  print(f"RMSE: {rmse_training:.4f}")

  # 3. PREDIZIONE E VALUTAZIONE
  y_pred = lr_model.predict(X_test)

  mae_test = mean_absolute_error(y_test, y_pred)
  mape_test = mean_absolute_percentage_error(y_test, y_pred) * 100
  r2_test = r2_score(y_test, y_pred)
  mse_test=mean_squared_error(y_test, y_pred)
  rmse_test = np.sqrt(mse_test)

  print(f"--- PERFORMANCE LINEAR REGRESSION (TESTING) ---")
  print(f"R^2 Score: {r2_test:.4f}")
  print(f"MAE: {mae_test:.4f}")
  print(f"MAPE: {mape_test:.2f}%")
  print(f"MSE: {mse_test:.4f}")
  print(f"RMSE: {rmse_test:.4f}")

  # --- ESPORTAZIONE CON M2CGEN ---

  filename_py=f"{MODEL_NAME}_1_{seed}.py"
  file_dimension_mpy=0

  try:

    python_code = m2c.export_to_python(lr_model)

    # Definizione del percorso del file
    file_path = MODELS_DIR / filename_py

    with open(file_path, "w") as f:
        f.write(python_code)
    print(f"Modello salvato con successo in: {file_path}")

    # Crea il percorso completo del file .py di origine usando la variabile filename
    py_file = MODELS_DIR / filename_py

    try:
        file_dimension_py=py_file.stat().st_size
    except:
      pass

    # Genera il percorso del file .mpy finale cambiando l'estensione (.with_suffix)
    # e mantenendo la cartella di destinazione
    mpy_file = MODELS_DIR / Path(filename_py).with_suffix(".mpy")

    print(f"Compilazione in corso: {py_file.name} -> {mpy_file.name}")

    # Esegue mpy-cross convertendo i percorsi in stringhe (richiesto da subprocess)
    subprocess.run(
        ["mpy-cross", "-o", str(mpy_file), str(py_file)],
        check=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )

    try:
      file_dimension_mpy=mpy_file.stat().st_size
    except:
      pass

  except subprocess.CalledProcessError as e:
      print(f"[ERRORE] Impossibile compilare {py_file.name}: {e.stderr.strip()}")
  except FileNotFoundError:
      print("[ERRORE CRITICO] Il comando 'mpy-cross' non è accessibile.")
  except Exception as e:
    print(f"Errore nell'esportazione MicroPython con m2cgen per {MODEL_NAME}: {e}")

  filename_c=f"{MODEL_NAME}_1_{seed}.h"
  file_dimension_c=0

  try:

    c_code = m2c.export_to_c(lr_model)

    # Definizione del percorso del file
    file_path = MODELS_DIR / filename_c

    with open(file_path, "w") as f:
        f.write(c_code)
    print(f"Modello salvato con successo in: {file_path}")


    try:
      file_dimension_c=file_path.stat().st_size
    except:
      pass

  except Exception as e:
    print(f"Errore nell'esportazione C con m2cgen per {MODEL_NAME}: {e}")


  info={
    "Modello": "LinearRegression",
    "Seed": str(seed),
    "Rank": 1,
    "Iperparametri": "-",
    "MAE_Kfold": "-",
    "R²_training": r2_training,
    "MAE_training": mae_training,
    "MAPE_training": f"{mape_training:.2f}%",
    "MSE_training": mse_training,
    "RMSE_training": rmse_training,
    "R²_test": r2_test,
    "MAE_test": mae_test,
    "MAPE_test": f"{mape_test:.2f}%",
    "MSE_test": mse_test,
    "RMSE_test": rmse_test,
    "Tempo di Inferenza": str(0),
    "Filename_py": filename_py,
    "Filename_mpy": filename_py.replace(".py",".mpy"),
    "Filename_c": filename_c,
    "DimensionePyhon": str(file_dimension_py),
    "DimensioneMicroPython": str(file_dimension_mpy),
    "DimensioneC": str(file_dimension_c)
  }

  df_info = pd.DataFrame([info])

  # 4. Salviamo in CSV usando i metodi nativi di Path
  # .exists() controlla se il file c'è già per decidere se fare l'append ('a') o la scrittura ('w')
  if BEST_MODELS_CSV.exists():
      df_info.to_csv(BEST_MODELS_CSV, mode='a', header=False, index=False)
  else:
      # Se il file non esiste, ci assicuriamo prima che la cartella di destinazione esista davvero
      df_info.to_csv(BEST_MODELS_CSV, mode='w', header=True, index=False)



=== INIZIO TRAINING LINEAR REGRESSION CON SEME 42 (1/10) ===

SHA256 Train set: 72d2e222075a8f8f1af5150136c80e70a1597099b54f860bb98ff9aa9b9de413
SHA256 Test set: c942b7fdd659fe737f68c6b07db1ce15dee591ea6dc65f83c9d58ac3038fa68c
Feature originali: 1206
Feature dopo PCA: 159
Training completato
--- PERFORMANCE LINEAR REGRESSION (TRAINING) ---
R^2 Score: 0.8516
MAE: 0.0452
MAPE: 13.87%
MSE: 0.0042
RMSE: 0.0645
--- PERFORMANCE LINEAR REGRESSION (TESTING) ---
R^2 Score: 0.8430
MAE: 0.0453
MAPE: 13.61%
MSE: 0.0043
RMSE: 0.0654
Modello salvato con successo in: [PROVA]ModelsToEvaluate/LinearRegression_1_42.py
Compilazione in corso: LinearRegression_1_42.py -> LinearRegression_1_42.mpy
Modello salvato con successo in: [PROVA]ModelsToEvaluate/LinearRegression_1_42.h


=== INIZIO TRAINING LINEAR REGRESSION CON SEME 8 (2/10) ===

SHA256 Train set: 083c606e79327ef028c51f1201017d6dfd4fb9c59e20c8d600e65df462083e0c
SHA256 Test set: da5c68fe723bb5105276aeee4ec0c11e18b67bf8d84fc22f0f3408e8dc44d32d
Feat